# Rats: A Normal Hierarchical Model

Fitting this model with **JuliaBUGS**, through the `mcmc` command line tool.

Every step below is one command: convert the graph to a model, fit it, check that it converged, draw the posterior, and package the run so it can be opened in the report app. Run the cells in order.

## Install

`mcmc` is a self-contained binary. `mcmc setup` then installs the JuliaBUGS toolchain, which on a fresh Colab runtime takes several minutes.

In [ ]:
!curl -fsSL https://mcmcjs.github.io/install.sh | sh
import os

os.environ["PATH"] = os.path.expanduser("~/.local/bin") + ":" + os.environ["PATH"]
!mcmc --version

In [ ]:
!mcmc setup --engine julia

## The graph

The model as it was drawn, with its data and initial values. Everything below is derived from this one document.

**To run your own model instead:** in the editor's Run tab press **Copy graph**, then replace the JSON below with what you copied. Nothing else in the notebook changes.

In [ ]:
# Replace this with your own graph: editor -> Run tab -> Copy graph.
graph = r'''
{
  "name": "Rats: A Normal Hierarchical Model",
  "elements": [
    {
      "id": "plate_i",
      "name": "Plate.i",
      "type": "node",
      "nodeType": "plate",
      "position": {
        "x": 342.5,
        "y": 350
      },
      "loopVariable": "i",
      "loopRange": "1:N"
    },
    {
      "id": "plate_j",
      "name": "Plate.j",
      "type": "node",
      "nodeType": "plate",
      "position": {
        "x": 514,
        "y": 355
      },
      "parent": "plate_i",
      "loopVariable": "j",
      "loopRange": "1:T"
    },
    {
      "id": "node_Y",
      "name": "Y",
      "type": "node",
      "nodeType": "observed",
      "position": {
        "x": 513,
        "y": 487
      },
      "parent": "plate_j",
      "distribution": "dnorm",
      "indices": "i,j",
      "observed": true,
      "param1": "mu",
      "param2": "tau.c"
    },
    {
      "id": "node_mu",
      "name": "mu",
      "type": "node",
      "nodeType": "deterministic",
      "position": {
        "x": 513,
        "y": 355
      },
      "parent": "plate_j",
      "equation": "alpha[i] + beta[i] * (x[j] - xbar)",
      "indices": "i,j"
    },
    {
      "id": "node_alpha",
      "name": "alpha",
      "type": "node",
      "nodeType": "stochastic",
      "position": {
        "x": 153,
        "y": 223
      },
      "parent": "plate_i",
      "distribution": "dnorm",
      "indices": "i",
      "param1": "alpha.c",
      "param2": "alpha.tau"
    },
    {
      "id": "node_beta",
      "name": "beta",
      "type": "node",
      "nodeType": "stochastic",
      "position": {
        "x": 343,
        "y": 223
      },
      "parent": "plate_i",
      "distribution": "dnorm",
      "indices": "i",
      "param1": "beta.c",
      "param2": "beta.tau"
    },
    {
      "id": "node_tau.c",
      "name": "tau.c",
      "type": "node",
      "nodeType": "stochastic",
      "position": {
        "x": 735,
        "y": 355
      },
      "distribution": "dgamma",
      "param1": "0.001",
      "param2": "0.001"
    },
    {
      "id": "node_sigma",
      "name": "sigma",
      "type": "node",
      "nodeType": "deterministic",
      "position": {
        "x": 735,
        "y": 487
      },
      "equation": "1 / sqrt(tau.c)"
    },
    {
      "id": "node_alpha.c",
      "name": "alpha.c",
      "type": "node",
      "nodeType": "stochastic",
      "position": {
        "x": 317,
        "y": 41
      },
      "distribution": "dnorm",
      "param1": "0.0",
      "param2": "1.0E-6"
    },
    {
      "id": "node_alpha.tau",
      "name": "alpha.tau",
      "type": "node",
      "nodeType": "stochastic",
      "position": {
        "x": 41,
        "y": 41
      },
      "distribution": "dgamma",
      "param1": "0.001",
      "param2": "0.001"
    },
    {
      "id": "node_beta.c",
      "name": "beta.c",
      "type": "node",
      "nodeType": "stochastic",
      "position": {
        "x": 581,
        "y": 41
      },
      "distribution": "dnorm",
      "param1": "0.0",
      "param2": "1.0E-6"
    },
    {
      "id": "node_beta.tau",
      "name": "beta.tau",
      "type": "node",
      "nodeType": "stochastic",
      "position": {
        "x": 449,
        "y": 41
      },
      "distribution": "dgamma",
      "param1": "0.001",
      "param2": "0.001"
    },
    {
      "id": "node_alpha0",
      "name": "alpha0",
      "type": "node",
      "nodeType": "deterministic",
      "position": {
        "x": 735,
        "y": 223
      },
      "equation": "alpha.c - xbar * beta.c"
    },
    {
      "id": "node_x",
      "name": "x",
      "type": "node",
      "nodeType": "constant",
      "position": {
        "x": 515,
        "y": 223
      },
      "indices": "j",
      "parent": "plate_j"
    },
    {
      "id": "node_xbar",
      "name": "xbar",
      "type": "node",
      "nodeType": "constant",
      "position": {
        "x": 821,
        "y": 41
      }
    },
    {
      "id": "edge_mu_to_y",
      "type": "edge",
      "source": "node_mu",
      "target": "node_Y"
    },
    {
      "id": "edge_tauc_to_y",
      "type": "edge",
      "source": "node_tau.c",
      "target": "node_Y"
    },
    {
      "id": "edge_alphai_to_mu",
      "type": "edge",
      "source": "node_alpha",
      "target": "node_mu"
    },
    {
      "id": "edge_betai_to_mu",
      "type": "edge",
      "source": "node_beta",
      "target": "node_mu"
    },
    {
      "id": "edge_x_to_mu",
      "type": "edge",
      "source": "node_x",
      "target": "node_mu"
    },
    {
      "id": "edge_xbar_to_mu",
      "type": "edge",
      "source": "node_xbar",
      "target": "node_mu"
    },
    {
      "id": "edge_alphac_to_alphai",
      "type": "edge",
      "source": "node_alpha.c",
      "target": "node_alpha"
    },
    {
      "id": "edge_alphatau_to_alphai",
      "type": "edge",
      "source": "node_alpha.tau",
      "target": "node_alpha"
    },
    {
      "id": "edge_betac_to_betai",
      "type": "edge",
      "source": "node_beta.c",
      "target": "node_beta"
    },
    {
      "id": "edge_betatau_to_betai",
      "type": "edge",
      "source": "node_beta.tau",
      "target": "node_beta"
    },
    {
      "id": "edge_tauc_to_sigma",
      "type": "edge",
      "source": "node_tau.c",
      "target": "node_sigma"
    },
    {
      "id": "edge_alphac_to_alpha0",
      "type": "edge",
      "source": "node_alpha.c",
      "target": "node_alpha0"
    },
    {
      "id": "edge_xbar_to_alpha0",
      "type": "edge",
      "source": "node_xbar",
      "target": "node_alpha0"
    },
    {
      "id": "edge_betac_to_alpha0",
      "type": "edge",
      "source": "node_beta.c",
      "target": "node_alpha0"
    }
  ],
  "dataContent": "{\\n  \\"data\\": {\\n    \\"N\\": 30,\\n    \\"T\\": 5,\\n    \\"x\\": [\\n      8,\\n      15,\\n      22,\\n      29,\\n      36\\n    ],\\n    \\"xbar\\": 22,\\n    \\"Y\\": [\\n      [\\n        151,\\n        199,\\n        246,\\n        283,\\n        320\\n      ],\\n      [\\n        145,\\n        199,\\n        249,\\n        293,\\n        354\\n      ],\\n      [\\n        147,\\n        214,\\n        263,\\n        312,\\n        328\\n      ],\\n      [\\n        155,\\n        200,\\n        237,\\n        272,\\n        297\\n      ],\\n      [\\n        135,\\n        188,\\n        230,\\n        280,\\n        323\\n      ],\\n      [\\n        159,\\n        210,\\n        252,\\n        298,\\n        331\\n      ],\\n      [\\n        141,\\n        189,\\n        231,\\n        275,\\n        305\\n      ],\\n      [\\n        159,\\n        201,\\n        248,\\n        297,\\n        338\\n      ],\\n      [\\n        177,\\n        236,\\n        285,\\n        350,\\n        376\\n      ],\\n      [\\n        134,\\n        182,\\n        220,\\n        260,\\n        296\\n      ],\\n      [\\n        160,\\n        208,\\n        261,\\n        313,\\n        352\\n      ],\\n      [\\n        143,\\n        188,\\n        220,\\n        273,\\n        314\\n      ],\\n      [\\n        154,\\n        200,\\n        244,\\n        289,\\n        325\\n      ],\\n      [\\n        171,\\n        221,\\n        270,\\n        326,\\n        358\\n      ],\\n      [\\n        163,\\n        216,\\n        242,\\n        281,\\n        312\\n      ],\\n      [\\n        160,\\n        207,\\n        248,\\n        288,\\n        324\\n      ],\\n      [\\n        142,\\n        187,\\n        234,\\n        280,\\n        316\\n      ],\\n      [\\n        156,\\n        203,\\n        243,\\n        283,\\n        317\\n      ],\\n      [\\n        157,\\n        212,\\n        259,\\n        307,\\n        336\\n      ],\\n      [\\n        152,\\n        203,\\n        246,\\n        286,\\n        321\\n      ],\\n      [\\n        154,\\n        205,\\n        253,\\n        298,\\n        334\\n      ],\\n      [\\n        139,\\n        190,\\n        225,\\n        267,\\n        302\\n      ],\\n      [\\n        146,\\n        191,\\n        229,\\n        272,\\n        302\\n      ],\\n      [\\n        157,\\n        211,\\n        250,\\n        285,\\n        323\\n      ],\\n      [\\n        132,\\n        185,\\n        237,\\n        286,\\n        331\\n      ],\\n      [\\n        160,\\n        207,\\n        257,\\n        303,\\n        345\\n      ],\\n      [\\n        169,\\n        216,\\n        261,\\n        295,\\n        333\\n      ],\\n      [\\n        157,\\n        205,\\n        248,\\n        289,\\n        316\\n      ],\\n      [\\n        137,\\n        180,\\n        219,\\n        258,\\n        291\\n      ],\\n      [\\n        153,\\n        200,\\n        244,\\n        286,\\n        324\\n      ]\\n    ]\\n  },\\n  \\"inits\\": {\\n    \\"alpha\\": [\\n      250,\\n      250,\\n      250,\\n      250,\\n      250,\\n      250,\\n      250,\\n      250,\\n      250,\\n      250,\\n      250,\\n      250,\\n      250,\\n      250,\\n      250,\\n      250,\\n      250,\\n      250,\\n      250,\\n      250,\\n      250,\\n      250,\\n      250,\\n      250,\\n      250,\\n      250,\\n      250,\\n      250,\\n      250,\\n      250\\n    ],\\n    \\"beta\\": [\\n      6,\\n      6,\\n      6,\\n      6,\\n      6,\\n      6,\\n      6,\\n      6,\\n      6,\\n      6,\\n      6,\\n      6,\\n      6,\\n      6,\\n      6,\\n      6,\\n      6,\\n      6,\\n      6,\\n      6,\\n      6,\\n      6,\\n      6,\\n      6,\\n      6,\\n      6,\\n      6,\\n      6,\\n      6,\\n      6\\n    ],\\n    \\"alpha.c\\": 150,\\n    \\"beta.c\\": 10,\\n    \\"tau.c\\": 1,\\n    \\"alpha.tau\\": 1,\\n    \\"beta.tau\\": 1\\n  }\\n}",
  "version": 1,
  "layout": {
    "showCodePanel": false,
    "codePanelWidth": 400,
    "codePanelHeight": 300,
    "showDataPanel": false,
    "dataPanelX": 40,
    "dataPanelY": 90,
    "dataPanelWidth": 400,
    "dataPanelHeight": 300
  }
}
'''

with open("model.json", "w") as f:
    f.write(graph)

import json

print(json.loads(graph).get("name", "model"), "written to model.json")

## The JuliaBUGS model

What the graph becomes as code, plus the spec that runs it.

In [ ]:
!mcmc convert model.json
!cat model.jl

## Fit

`mcmc run` does the whole workflow: it samples, checks convergence, and records the run so the later commands can find it.

In [ ]:
!mcmc run model.toml --chains 2 --draws 1000 --warmup 1000 --seed 42

## Did it converge?

R-hat near 1 and a healthy effective sample size per parameter. `mcmc diagnose` exits non-zero if it did not, so this is the cell to trust before reading the posterior.

In [ ]:
!mcmc summary
!mcmc diagnose

## Plots

Traces and ranks show the chains mixing; densities and the forest plot show the posterior itself.

In [ ]:
from IPython.display import SVG, display

for kind in ["trace","density","forest","rank"]:
    !mcmc plot --kind {kind} --format svg -o {kind}.svg
    print(kind)
    display(SVG(f"{kind}.svg"))

## Open the run in the report app

A run bundle holds the samples, the spec and the diagnostics in one file. Download it, then drop it into [the report app](https://mcmcjs.github.io/report/) to explore every parameter interactively.

In [ ]:
!mcmc export bundle -o run.mcmcrun.json

from google.colab import files  # skip this line outside Colab

files.download("run.mcmcrun.json")